# Code to classify the neurons into different subtypes: Visual, Memory-delay/Visuo-motor, Motor neurons 
- Neurons were classified by using the data from the MGS saccade task. 
- Classification into neuronal subtypes was done based on MI and permutation test for significance

The current code works for MGS data with distractors. Needs to be adapted for without distractor. 
This data is further being used for Functional Group and Timescale comparison


## How to run
1. Pick the session config in the **settings cell** below (or set the `CCLAB_CONFIG` environment variable).
   Every path, analysis window and statistics setting comes from the `[mi_classification]` section of that file —
   don't edit paths in the cells further down. See `docs/configuration.md`.
2. Run all cells (`uv run jupyter lab`, then *Run > Run All Cells*).
3. Results and figures go to `out_dir`; a copy of the config + git commit is archived in `out_dir/_runs/`.


In [ ]:
# ============================ SETTINGS ============================
# Everything that differs between experiments lives in the session config.
import os
CONFIG = os.environ.get("CCLAB_CONFIG", "configs/npx/venni_20250611_SEF.toml")
# ==================================================================

from cclab_preproc import load_config, write_run_record

cfg = load_config(CONFIG)
S = cfg.step("mi_classification")   # settings with paths resolved
print(cfg.summary())

N_JOBS = S["n_jobs"]                  # parallel workers (-1 = all cores)
N_PERMUTATIONS = S["n_permutations"]  # shuffles per permutation test
N_NEIGHBORS = S["n_neighbors"]        # k for the mutual-information estimator
ALPHA = S["alpha"]                    # significance threshold
MIN_RATE = S["min_rate"]              # minimum mean response in the epoch window (Hz)


In [ ]:
from sklearn.feature_selection import mutual_info_classif
import numpy as np
from joblib import Parallel, delayed
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import os
from scipy.stats import vonmises
from scipy.optimize import minimize

# Compute mutual information for individual neurons

def neuron_mi_perm(neuron_data, labels, n_neighbors=5, n_permutations=1000):
    
    actual_mi = mutual_info_classif(neuron_data[:, None], labels, n_neighbors=n_neighbors)[0]
    
    permuted_mis = np.array([
        mutual_info_classif(neuron_data[:, None], np.random.permutation(labels), n_neighbors=n_neighbors)[0]
        for _ in range(n_permutations)
    ])
    
    p_val = np.mean(permuted_mis >= actual_mi)
    return actual_mi, p_val



In [ ]:
# Compute mutual information difference between visual and baseline epochs

def mi_diff_permutation_test(baseline, visual, labels, n_neighbors=5, n_permutations=1000):
    # Compute actual difference in MI
    mi_visual = mutual_info_classif(visual[:, None], labels, n_neighbors=n_neighbors)[0]
    mi_baseline = mutual_info_classif(baseline[:, None], labels, n_neighbors=n_neighbors)[0]
    mi_diff = mi_visual - mi_baseline

    # Permutation test: shuffle labels and recompute MI diff
    perm_diffs = np.array([
        mutual_info_classif(visual[:, None], np.random.permutation(labels), n_neighbors=n_neighbors)[0] -
        mutual_info_classif(baseline[:, None], np.random.permutation(labels), n_neighbors=n_neighbors)[0]
        for _ in range(n_permutations)
    ])

    # One-tailed p-value: how often permuted MI_diff >= observed MI_diff
    p_val = np.mean(np.abs(perm_diffs) >= np.abs(mi_diff))
    return mi_diff, p_val


In [ ]:

import numpy as np
from sklearn.feature_selection import mutual_info_classif
from statsmodels.stats.multitest import multipletests

def mi_responsiveness_test(baseline_fr_list, 
                           task_fr_list, 
                           n_neighbors=5, 
                           n_permutations=1000,
                           min_mi_threshold=0.01):
    """
    Tests if a neuron is *responsive* using MI, with a built-in
    effect size threshold to save computation.

    It checks if firing rate contains information about 
    *which epoch* the data came from (baseline vs. task).

    Args:
    - baseline_fr_list (array): 1D array (n_trials,) of avg FR in baseline
    - task_fr_list (array): 1D array (n_trials,) of avg FR in task epoch
    - n_neighbors (int): k-neighbors for MI estimation
    - n_permutations (int): Number of shuffles for the p-value
    - min_mi_threshold (float): The "effect size" filter. If actual_mi is
                                below this, the test stops and returns p=1.0.

    Returns:
    - actual_mi (float): The calculated Mutual Information (the effect size)
    - p_val (float): The p-value from the permutation test
    """
    
    # 1. Combine data and create new labels (0=baseline, 1=task)
    n_baseline = len(baseline_fr_list)
    n_task = len(task_fr_list)
    
    combined_fr = np.concatenate((baseline_fr_list, task_fr_list))
    epoch_labels = np.concatenate((np.zeros(n_baseline), np.ones(n_task)))
    
    # 2. Calculate the actual MI (the effect size)
    # Reshape combined_fr to (n_samples, 1) for the function
    actual_mi = mutual_info_classif(
        combined_fr[:, None], 
        epoch_labels, 
        n_neighbors=n_neighbors,
        random_state=42 # for reproducibility
    )[0]
    
    # 3. Apply Effect Size Filter
    # If the effect is already below our threshold,
    # return it with a non-significant p-value (1.0).
    if actual_mi < min_mi_threshold:
        return actual_mi, 1.0
    
    # 4. Build the null distribution (only if MI is large enough)
    permuted_mis = np.zeros(n_permutations)
    for i in range(n_permutations):
        shuffled_labels = np.random.permutation(epoch_labels)
        permuted_mis[i] = mutual_info_classif(
            combined_fr[:, None], 
            shuffled_labels, 
            n_neighbors=n_neighbors,
            random_state=i # different seed for each perm
        )[0]
        
    # 5. Calculate the p-value
    # How often is a random MI >= our actual MI?
    p_val = np.mean(permuted_mis >= actual_mi)
    
    return actual_mi, p_val

In [ ]:
import numpy as np
# We use LinearDiscriminantAnalysis: it's a simple, fast, and powerful classifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

def classifier_responsiveness_test(baseline_fr_list, 
                                   task_fr_list, 
                                   n_permutations=1000, 
                                   n_cv_folds=5):
    """
    Tests for neural responsiveness using a binary classifier's accuracy.

    It checks how accurately a classifier can predict the epoch
    (baseline vs. task) given the firing rate.

    Args:
    - baseline_fr_list (array): 1D array (n_trials,) of avg FR in baseline
    - task_fr_list (array): 1D array (n_trials,) of avg FR in task epoch
    - n_permutations (int): Number of shuffles for the p-value
    - n_cv_folds (int): Number of folds for cross-validation

    Returns:
    - actual_accuracy (float): The cross-validated accuracy (your index)
    - p_val (float): The p-value from the permutation test
    """
    
    # 1. Combine data and create labels (0=baseline, 1=task)
    n_baseline = len(baseline_fr_list)
    n_task = len(task_fr_list)
    
    combined_fr = np.concatenate((baseline_fr_list, task_fr_list))
    epoch_labels = np.concatenate((np.zeros(n_baseline), np.ones(n_task)))
    
    # Reshape X for sklearn: (n_samples, n_features)
    X = combined_fr[:, None]
    y = epoch_labels
    
    # 2. Initialize the classifier
    model = LogisticRegression()
    
    # 3. Calculate the actual accuracy using cross-validation
    actual_scores = cross_val_score(model, X, y, cv=n_cv_folds, scoring='accuracy')
    actual_accuracy = np.mean(actual_scores)
    
    # 4. Build the null distribution
    permuted_accuracies = np.zeros(n_permutations)
    for i in range(n_permutations):
        shuffled_labels = np.random.permutation(y)
        
        # Get accuracy for this shuffled permutation
        perm_scores = cross_val_score(model, X, shuffled_labels, cv=n_cv_folds, scoring='accuracy')
        permuted_accuracies[i] = np.mean(perm_scores)
        
    # 5. Calculate the p-value
    # How many shuffled accuracies were >= our actual accuracy?
    n_extreme = np.sum(permuted_accuracies >= actual_accuracy)
    
    # Use the (n+1)/(N+1) method to avoid p=0
    p_val = (n_extreme + 1) / (n_permutations + 1)
    
    return actual_accuracy, p_val



In [ ]:
def align_psth_to_saccade(psth_data, saccade_times, time_bins, pre_ms=400, post_ms=400):
    """
    Aligns PSTHs (binned data) to saccade onset using bin-level indexing.

    Args:
        psth_data (ndarray): shape (n_trials, n_elecs, n_timepoints), aligned to target onset
        saccade_times (ndarray): shape (n_trials,), saccade times in seconds relative to target
        time_bins (ndarray): shape (n_timepoints,), bin centers in ms (e.g., from -500 to 1790 in steps of 10)
        pre_ms (int): time before saccade to include (default 300ms)
        post_ms (int): time after saccade to include (default 300ms)

    Returns:
        psth_sacc (ndarray): shape (n_trials, n_elecs, n_bins), aligned to saccade
    """
    n_neurons, n_trials, T = psth_data.shape
    bin_size = time_bins[1] - time_bins[0]  # should be 10
    total_bins = (pre_ms + post_ms) // bin_size  # e.g., 600 ms / 10 = 60
    psth_sacc = np.full((n_neurons, n_trials, total_bins), np.nan)

    for i in range(n_trials):
        # Convert saccade time (in sec) to ms, then find closest bin
        saccade_time_ms = saccade_times[i] * 1000
        saccade_bin = np.argmin(np.abs(time_bins - saccade_time_ms))

        # Define bin window around saccade
        start_bin = saccade_bin - (pre_ms // bin_size)
        end_bin = saccade_bin + (post_ms // bin_size)

        # Define valid source and destination indices
        src_start = max(0, start_bin)
        src_end = min(end_bin, T)
        dst_start = max(0, -start_bin)
        dst_end = dst_start + (src_end - src_start)

        # Copy data into aligned PSTH
        psth_sacc[:, i, dst_start:dst_end] = psth_data[:, i, src_start:src_end]

    return psth_sacc

In [ ]:
# Analysis windows in PSTH bins (start, stop), set in the config
baseline_window = slice(*S["baseline_bins"])
visual_window = slice(*S["visual_bins"])
memory_window = slice(*S["memory_bins"])


In [ ]:
import os
filepath = S["psth_dir"]   # input: psth_3d_*.mat, angsSuccess/distractor/good_neurons .npy, tau_values.mat
mat_path = S["mat_dir"]    # input: SaccadeTimes_*.mat
out_dir = S["out_dir"]     # output: results + figures
os.makedirs(out_dir, exist_ok=True)
write_run_record(cfg, out_dir, "mi_classification")

# Loading the psth_3d .mat file
import scipy.io as sio
def load_psth_data(file_path):
    data = sio.loadmat(file_path)
    return data['psth_3d']

psth_0_mat = load_psth_data(os.path.join(filepath, S["psth_files"]["0"]))
#psth_45_mat = load_psth_data(os.path.join(filepath, S["psth_files"]["45"]))
#psth_180_mat = load_psth_data(os.path.join(filepath, S["psth_files"]["180"]))

angsSuccess = np.load(os.path.join(filepath, S["angs_name"]))
distractor = np.load(os.path.join(filepath, S["distractor_name"]))
good_neurons = np.load(os.path.join(filepath, S["good_neurons_name"]))

# Detecting Visual Neurons 

## MI calculated for visual epoch

In [ ]:
# Stacking PSTH for all 0, 45, and 180
psth_combined = np.concatenate([psth_0_mat], axis = 1)

psth_comb_visual = np.mean(psth_combined[:,:,visual_window], axis = 2)
psth_comb_baseline = np.mean(psth_combined[:,:, baseline_window], axis = 2)

results_combined = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(neuron_mi_perm)(psth_comb_visual[i], angsSuccess, n_neighbors=N_NEIGHBORS, n_permutations=N_PERMUTATIONS)
    for i in range(psth_combined.shape[0])
)



## MI calculated for visual - baseline epoch

In [ ]:


results_combined_comp = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(mi_diff_permutation_test)(psth_comb_baseline[i],psth_comb_visual[i], angsSuccess, n_neighbors=N_NEIGHBORS, n_permutations=N_PERMUTATIONS)
    for i in range(psth_combined.shape[0])
)


Extracting values to detect visual neurons

In [ ]:
# Unpack results
mi_comb, p_comb = zip(*results_combined)
mi_comb = np.array(mi_comb)
p_comb = np.array(p_comb)

# Unpack results
mi_comp, p_comp = zip(*results_combined_comp)
mi_comp = np.array(mi_comp)
p_comp = np.array(p_comp)

visual_neurons_all = np.array(np.where( (p_comb < ALPHA)  | (p_comp < ALPHA))[0])
print(f"Number of visual neurons: {visual_neurons_all.shape[0]}")

# Find visual neurons that have a spiking rate of less than 5 Hz in the visual window and delete them from both 180 and 45 degree visual neurons
visual_neurons_psth = psth_combined[visual_neurons_all, : ,: ]
mean_fr = np.nanmean(np.nanmean(visual_neurons_psth[:, :, visual_window], axis=2), axis = 1)

visual_neurons_final_idx = np.where((mean_fr > MIN_RATE))[0]
print(f"Number of visual neurons (> MIN_RATE): {visual_neurons_final_idx.shape[0]}")

visual_neurons_final = visual_neurons_all[visual_neurons_final_idx]

In [ ]:
visual_neurons_psth_0_new = psth_0_mat[visual_neurons_final,:,:]
visual_neurons_psth_45_new = psth_45_mat[visual_neurons_final,:,:]
visual_neurons_psth_180_new = psth_180_mat[visual_neurons_final,:,:]

## Plot all the visual neurons for each distractor type

In [ ]:
#Plot only the visual neurons psth for 45 and 180 degrees
#Plot them as 2 different subplots for each neuron. And also plot each direction in a different color
# Make sure that the firing rate is equal on both the subplots such that the y-axis has the value of the maximum firing rate between the two subplots
from pathlib import Path

window = [S["psth_start_ms"] / 1000, S["psth_end_ms"] / 1000]
bin_width = 0.01  # 10 ms bin width
bin_edges = np.arange(window[0], window[1] + bin_width, bin_width)
bin_edges = bin_edges[:-1]  # Center the bins
for neuron_idx in range(len(visual_neurons_final)):
    print(neuron_idx)
    neuron_id = visual_neurons_final[neuron_idx]
    print(f'Plotting PSTH for Neuron {neuron_id}...')
    psth_0_temp = visual_neurons_psth_0_new[neuron_idx,:,:]
    psth_45_temp = visual_neurons_psth_45_new[neuron_idx,:, :]
    psth_180_temp = visual_neurons_psth_180_new[neuron_idx,:, :]

    # Calculate the maximum firing rate across all groups for both distractors
    max_firing_rate_0 = max([np.max(np.mean(psth_0_temp[np.where(angsSuccess[np.where(distractor == 0.0)[0]] == group)[0], :], axis=0)) 
                              for group in np.unique(angsSuccess[np.where(distractor == 0.0)[0]])])
    max_firing_rate_45 = max([np.max(np.mean(psth_45_temp[np.where(angsSuccess[np.where(distractor == 45.0)[0]] == group)[0], :], axis=0)) 
                              for group in np.unique(angsSuccess[np.where(distractor == 45.0)[0]])])
    max_firing_rate_180 = max([np.max(np.mean(psth_180_temp[np.where(angsSuccess[np.where(distractor == 180.0)[0]] == group)[0], :], axis=0)) 
                               for group in np.unique(angsSuccess[np.where(distractor == 180.0)[0]])])
    max_firing_rate = max(max_firing_rate_0,max_firing_rate_45, max_firing_rate_180)

    plt.figure(figsize=(12, 6))
    # Plot for 0° distractor
    plt.subplot(1, 3, 1)
    for group in np.unique(angsSuccess[np.where(distractor == 0.0)[0]]):
        group_indices = np.where(angsSuccess[np.where(distractor == 0.0)[0]] == group)[0]
        if len(group_indices) > 0:
            group_mean = np.mean(psth_0_temp[group_indices, :], axis=0)
            plt.plot(bin_edges * 1000, group_mean, label=f'Group {group}')
    plt.title(f'Neuron {neuron_id} - PSTH for 0° Distractor')
    plt.xlabel('Time (ms)')
    plt.ylabel('Firing Rate (Hz)')
    plt.ylim(0, max_firing_rate + 5)  # Set y-axis limit
    plt.axvspan(0, 50, color='gray', alpha=0.3)
    plt.legend()

    # Plot for 45° distractor
    plt.subplot(1, 3, 2)
    for group in np.unique(angsSuccess[np.where(distractor == 45.0)[0]]):
        group_indices = np.where(angsSuccess[np.where(distractor == 45.0)[0]] == group)[0]
        if len(group_indices) > 0:
            group_mean = np.mean(psth_45_temp[group_indices, :], axis=0)
            plt.plot(bin_edges * 1000, group_mean, label=f'Group {group}')
    plt.title(f'Neuron {neuron_id} - PSTH for 45° Distractor')
    plt.xlabel('Time (ms)')
    plt.ylabel('Firing Rate (Hz)')
    plt.ylim(0, max_firing_rate + 5)  # Set y-axis limit
    plt.axvspan(0, 50, color='gray', alpha=0.3)
    plt.legend()

    # Plot for 180° distractor
    plt.subplot(1, 3, 3)
    for group in np.unique(angsSuccess[np.where(distractor == 180.0)[0]]):
        group_indices = np.where(angsSuccess[np.where(distractor == 180.0)[0]] == group)[0]
        if len(group_indices) > 0:
            group_mean = np.mean(psth_180_temp[group_indices, :], axis=0)
            plt.plot(bin_edges * 1000, group_mean, label=f'Group {group}')
    plt.title(f'Neuron {neuron_id} - PSTH for 180° Distractor ')
    plt.xlabel('Time (ms)')
    plt.ylabel('Firing Rate (Hz)')
    plt.ylim(0, max_firing_rate + 5)  # Set y-axis limit
    plt.axvspan(0, 50, color='gray', alpha=0.3)
    plt.legend()
    # Save the figure
    plt.tight_layout()
    new_path = Path(out_dir) / 'PSTH_Visual_neurons_all'
    new_path.mkdir(parents=True, exist_ok=True)
    plt.savefig(new_path / f'VNeuron_{neuron_id}_PSTH_all.png', dpi=300)

    plt.show()

    plt.close()


## Individual MI for each distractor type visual vs baseline

In [ ]:
# Detecting all the visual neurons by comparing to baseline
psth_0_baseline = np.mean(psth_0_mat[:,:,baseline_window], axis = 2)
psth_45_baseline = np.mean(psth_45_mat[:, :, baseline_window], axis=2)
psth_180_baseline = np.mean(psth_180_mat[:, :, baseline_window], axis=2)

psth_0 = np.mean(psth_0_mat[:,:, visual_window], axis = 2)
psth_45 = np.mean(psth_45_mat[:,:, visual_window], axis=2)
psth_180 = np.mean(psth_180_mat[:,:, visual_window], axis=2)
n_neurons, n_trials = psth_180.shape

results_0 = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(mi_diff_permutation_test)(psth_0_baseline[i], psth_0[i], angsSuccess[np.where(distractor == 0.0)[0]], n_neighbors=N_NEIGHBORS, n_permutations=N_PERMUTATIONS)
    for i in range(psth_0.shape[0])
)
# psth_180 shape: (n_neurons, n_trials)
# angs_45 shape: (n_trials,)
results_45 = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(mi_diff_permutation_test)(psth_45_baseline[i], psth_45[i], angsSuccess[np.where(distractor == 45.0)[0]], n_neighbors=N_NEIGHBORS, n_permutations=N_PERMUTATIONS)
    for i in range(psth_45.shape[0])
)

results_180 = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(mi_diff_permutation_test)(psth_180_baseline[i], psth_180[i], angsSuccess[np.where(distractor == 180.0)[0]], n_neighbors=N_NEIGHBORS, n_permutations=N_PERMUTATIONS)
    for i in range(psth_180.shape[0])
)
# Unpack results
mi_0_vcomp, p_0_vcomp = zip(*results_0)
mi_0_vcomp = np.array(mi_0_vcomp)
p_0_vcomp = np.array(p_0_vcomp)

# Unpack results
mi_45_vcomp, p_45_vcomp = zip(*results_45)
mi_45_vcomp = np.array(mi_45_vcomp)
p_45_vcomp = np.array(p_45_vcomp)

# Unpack results
mi_180_vcomp, p_180_vcomp = zip(*results_180)
mi_180_vcomp = np.array(mi_180_vcomp)
p_180_vcomp = np.array(p_180_vcomp)


## Individual MI for each distractor type visual only

In [ ]:

results_0 = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(neuron_mi_perm)(psth_0[i], angsSuccess[np.where(distractor == 0.0)[0]], n_neighbors=N_NEIGHBORS, n_permutations=N_PERMUTATIONS)
    for i in range(psth_0.shape[0])
)
# psth_180 shape: (n_neurons, n_trials)
# angs_45 shape: (n_trials,)
results_45 = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(neuron_mi_perm)(psth_45[i], angsSuccess[np.where(distractor == 45.0)[0]], n_neighbors=N_NEIGHBORS, n_permutations=N_PERMUTATIONS)
    for i in range(psth_45.shape[0])
)

results_180 = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(neuron_mi_perm)(psth_180[i], angsSuccess[np.where(distractor == 180.0)[0]], n_neighbors=N_NEIGHBORS, n_permutations=N_PERMUTATIONS)
    for i in range(psth_45.shape[0])
)

# Unpack results
mi_0, p_0 = zip(*results_45)
mi_0 = np.array(mi_0)
p_0 = np.array(p_0)

# Unpack results
mi_45, p_45 = zip(*results_45)
mi_45 = np.array(mi_45)
p_45 = np.array(p_45)

# Unpack results
mi_180, p_180 = zip(*results_180)
mi_180 = np.array(mi_180)
p_180 = np.array(p_180)


# Detecting memory delay neurons

## MI calculated for memory epoch

In [ ]:


psth_comb_memory= np.mean(psth_combined[:,:,memory_window], axis = 2)


results_combined = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(neuron_mi_perm)(psth_comb_memory[i], angsSuccess, n_neighbors=N_NEIGHBORS, n_permutations=N_PERMUTATIONS)
    for i in range(psth_combined.shape[0])
)



## MI calculated for memory vs baseline epoch

In [ ]:


results_combined_comp = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(mi_diff_permutation_test)(psth_comb_baseline[i],psth_comb_memory[i], angsSuccess, n_neighbors=N_NEIGHBORS, n_permutations=N_PERMUTATIONS)
    for i in range(psth_combined.shape[0])
)


Extracting values to detect memory neurons

In [ ]:


# Unpack results
mi_comb_mem, p_comb_mem = zip(*results_combined)
mi_comb_mem = np.array(mi_comb_mem)
p_comb_mem = np.array(p_comb_mem)

# Unpack results
mi_comp_mem, p_comp_mem = zip(*results_combined_comp)
mi_comp_mem = np.array(mi_comp_mem)
p_comp_mem = np.array(p_comp_mem)


In [ ]:
memory_neurons_all = np.array(np.where( (p_comb_mem < ALPHA)  | (p_comp_mem < ALPHA))[0])
print(f"Number of memory neurons: {memory_neurons_all.shape[0]}")

# Find visual neurons that have a spiking rate of less than 5 Hz in the visual window and delete them from both 180 and 45 degree visual neurons
memory_neurons_psth = psth_combined[memory_neurons_all, : ,: ]
mean_fr = np.nanmean(np.nanmean(memory_neurons_psth[:, :, memory_window], axis=2), axis = 1)

memory_neurons_final_idx = np.where((mean_fr > MIN_RATE))[0]
print(f"Number of memory neurons (> MIN_RATE): {memory_neurons_final_idx.shape[0]}")

memory_neurons_final = memory_neurons_all[memory_neurons_final_idx]

In [ ]:
memory_neurons_psth_0_new = psth_0_mat[memory_neurons_final,:,:]
memory_neurons_psth_45_new = psth_45_mat[memory_neurons_final,:,:]
memory_neurons_psth_180_new = psth_180_mat[memory_neurons_final,:,:]

## Plot all the memory-delay neurons

In [ ]:
#Plot only the visual neurons psth for 45 and 180 degrees
#Plot them as 2 different subplots for each neuron. And also plot each direction in a different color
# Make sure that the firing rate is equal on both the subplots such that the y-axis has the value of the maximum firing rate between the two subplots
from pathlib import Path

window = [S["psth_start_ms"] / 1000, S["psth_end_ms"] / 1000]
bin_width = 0.01  # 10 ms bin width
bin_edges = np.arange(window[0], window[1] + bin_width, bin_width)
bin_edges = bin_edges[:-1]  # Center the bins
for neuron_idx in range(len(memory_neurons_final)):
    print(neuron_idx)
    neuron_id = memory_neurons_final[neuron_idx]
    print(f'Plotting PSTH for Neuron {neuron_id}...')
    psth_0_temp = memory_neurons_psth_0_new[neuron_idx,:,:]
    psth_45_temp = memory_neurons_psth_45_new[neuron_idx,:, :]
    psth_180_temp = memory_neurons_psth_180_new[neuron_idx,:, :]

    # Calculate the maximum firing rate across all groups for both distractors
    max_firing_rate_0 = max([np.max(np.mean(psth_0_temp[np.where(angsSuccess[np.where(distractor == 0.0)[0]] == group)[0], :], axis=0)) 
                              for group in np.unique(angsSuccess[np.where(distractor == 0.0)[0]])])
    max_firing_rate_45 = max([np.max(np.mean(psth_45_temp[np.where(angsSuccess[np.where(distractor == 45.0)[0]] == group)[0], :], axis=0)) 
                              for group in np.unique(angsSuccess[np.where(distractor == 45.0)[0]])])
    max_firing_rate_180 = max([np.max(np.mean(psth_180_temp[np.where(angsSuccess[np.where(distractor == 180.0)[0]] == group)[0], :], axis=0)) 
                               for group in np.unique(angsSuccess[np.where(distractor == 180.0)[0]])])
    max_firing_rate = max(max_firing_rate_0,max_firing_rate_45, max_firing_rate_180)

    plt.figure(figsize=(12, 6))
    # Plot for 0° distractor
    plt.subplot(1, 3, 1)
    for group in np.unique(angsSuccess[np.where(distractor == 0.0)[0]]):
        group_indices = np.where(angsSuccess[np.where(distractor == 0.0)[0]] == group)[0]
        if len(group_indices) > 0:
            group_mean = np.mean(psth_0_temp[group_indices, :], axis=0)
            plt.plot(bin_edges * 1000, group_mean, label=f'Group {group}')
    plt.title(f'Neuron {neuron_id} - PSTH for 0° Distractor')
    plt.xlabel('Time (ms)')
    plt.ylabel('Firing Rate (Hz)')
    plt.ylim(0, max_firing_rate + 5)  # Set y-axis limit
    plt.axvspan(0, 50, color='gray', alpha=0.3)
    plt.legend()

    # Plot for 45° distractor
    plt.subplot(1, 3, 2)
    for group in np.unique(angsSuccess[np.where(distractor == 45.0)[0]]):
        group_indices = np.where(angsSuccess[np.where(distractor == 45.0)[0]] == group)[0]
        if len(group_indices) > 0:
            group_mean = np.mean(psth_45_temp[group_indices, :], axis=0)
            plt.plot(bin_edges * 1000, group_mean, label=f'Group {group}')
    plt.title(f'Neuron {neuron_id} - PSTH for 45° Distractor')
    plt.xlabel('Time (ms)')
    plt.ylabel('Firing Rate (Hz)')
    plt.ylim(0, max_firing_rate + 5)  # Set y-axis limit
    plt.axvspan(0, 50, color='gray', alpha=0.3)
    plt.legend()

    # Plot for 180° distractor
    plt.subplot(1, 3, 3)
    for group in np.unique(angsSuccess[np.where(distractor == 180.0)[0]]):
        group_indices = np.where(angsSuccess[np.where(distractor == 180.0)[0]] == group)[0]
        if len(group_indices) > 0:
            group_mean = np.mean(psth_180_temp[group_indices, :], axis=0)
            plt.plot(bin_edges * 1000, group_mean, label=f'Group {group}')
    plt.title(f'Neuron {neuron_id} - PSTH for 180° Distractor ')
    plt.xlabel('Time (ms)')
    plt.ylabel('Firing Rate (Hz)')
    plt.ylim(0, max_firing_rate + 5)  # Set y-axis limit
    plt.axvspan(0, 50, color='gray', alpha=0.3)
    plt.legend()
    # Save the figure
    plt.tight_layout()
    new_path = Path(out_dir) / 'PSTH_Memory_neurons_all'
    new_path.mkdir(parents=True, exist_ok=True)
    plt.savefig(new_path / f'MemNeuron_{neuron_id}_PSTH_all.png', dpi=300)

    plt.show()

    plt.close()


## Individual MI for memory vs baseline epoch for each distractor

In [ ]:

psth_0 = np.mean(psth_0_mat[:,:, memory_window], axis = 2)
psth_45 = np.mean(psth_45_mat[:,:, memory_window], axis=2)
psth_180 = np.mean(psth_180_mat[:,:, memory_window], axis=2)
n_neurons, n_trials = psth_180.shape

results_0 = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(mi_diff_permutation_test)(psth_0_baseline[i], psth_0[i], angsSuccess[np.where(distractor == 0.0)[0]], n_neighbors=N_NEIGHBORS, n_permutations=N_PERMUTATIONS)
    for i in range(psth_0.shape[0])
)
# psth_180 shape: (n_neurons, n_trials)
# angs_45 shape: (n_trials,)
results_45 = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(mi_diff_permutation_test)(psth_45_baseline[i], psth_45[i], angsSuccess[np.where(distractor == 45.0)[0]], n_neighbors=N_NEIGHBORS, n_permutations=N_PERMUTATIONS)
    for i in range(psth_45.shape[0])
)

results_180 = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(mi_diff_permutation_test)(psth_180_baseline[i], psth_180[i], angsSuccess[np.where(distractor == 180.0)[0]], n_neighbors=N_NEIGHBORS, n_permutations=N_PERMUTATIONS)
    for i in range(psth_180.shape[0])
)


In [ ]:
# Unpack results
mi_0_memcomp, p_0_memcomp = zip(*results_0)
mi_0_memcomp = np.array(mi_0_memcomp)
p_0_memcomp = np.array(p_0_memcomp)

# Unpack results
mi_45_memcomp, p_45_memcomp = zip(*results_45)
mi_45_memcomp = np.array(mi_45_memcomp)
p_45_memcomp = np.array(p_45_memcomp)

# Unpack results
mi_180_memcomp, p_180_memcomp = zip(*results_180)
mi_180_memcomp = np.array(mi_180_memcomp)
p_180_memcomp = np.array(p_180_memcomp)


## Individual MI for memory epoch for each distractor

In [ ]:

results_0 = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(neuron_mi_perm)(psth_0[i], angsSuccess[np.where(distractor == 0.0)[0]], n_neighbors=N_NEIGHBORS, n_permutations=N_PERMUTATIONS)
    for i in range(psth_0.shape[0])
)
# psth_180 shape: (n_neurons, n_trials)
# angs_45 shape: (n_trials,)
results_45 = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(neuron_mi_perm)(psth_45[i], angsSuccess[np.where(distractor == 45.0)[0]], n_neighbors=N_NEIGHBORS, n_permutations=N_PERMUTATIONS)
    for i in range(psth_45.shape[0])
)

results_180 = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(neuron_mi_perm)(psth_180[i], angsSuccess[np.where(distractor == 180.0)[0]], n_neighbors=N_NEIGHBORS, n_permutations=N_PERMUTATIONS)
    for i in range(psth_45.shape[0])
)

# Unpack results
mi_0_mem, p_0_mem = zip(*results_45)
mi_0_mem = np.array(mi_0_mem)
p_0_mem = np.array(p_0_mem)

# Unpack results
mi_45_mem, p_45_mem = zip(*results_45)
mi_45_mem = np.array(mi_45_mem)
p_45_mem = np.array(p_45_mem)

# Unpack results
mi_180_mem, p_180_mem = zip(*results_180)
mi_180_mem = np.array(mi_180_mem)
p_180_mem = np.array(p_180_mem)


# Detecting motor neurons

## Aligning psth data to saccade onset

In [ ]:
import scipy.io as sio
saccadeTimes = sio.loadmat(os.path.join(mat_path, S["saccade_files"]["0"]))['saccadeTimes']
saccadeTimes45 = sio.loadmat(os.path.join(mat_path, S["saccade_files"]["45"]))['saccadeTimes']
saccadeTimes180 = sio.loadmat(os.path.join(mat_path, S["saccade_files"]["180"]))['saccadeTimes']
saccadeTimes = saccadeTimes[0] + S["saccade_offset_s"] # Saccade times are after target onset, so add 0.9s to get time from fixation onset in seconds
saccadeTimes45 = saccadeTimes45[0] + S["saccade_offset_s"] # Saccade times are after target onset, so add 0.9s to get time from fixation onset in seconds
saccadeTimes180 = saccadeTimes180[0] + S["saccade_offset_s"] # Saccade times are after target onset, so add 0.9s to get time from fixation onset in seconds

In [ ]:
time_bins = np.arange(S["psth_start_ms"], S["psth_end_ms"], S["bin_ms"])  # PSTH bin start times (ms)
SACC_PRE, SACC_POST = S["sacc_pre_ms"], S["sacc_post_ms"]  # saccade-aligned window
align_psth_Saccade = align_psth_to_saccade(psth_0_mat, saccadeTimes, time_bins, pre_ms=SACC_PRE, post_ms=SACC_POST)
align_psth_Saccade45 = align_psth_to_saccade(psth_45_mat, saccadeTimes45, time_bins, pre_ms=SACC_PRE, post_ms=SACC_POST)
align_psth_Saccade180 = align_psth_to_saccade(psth_180_mat, saccadeTimes180, time_bins, pre_ms=SACC_PRE, post_ms=SACC_POST)

Plot no distractor

In [ ]:


# Plot data aligned to saccade for each neuron and trial averaged across each angSuccess trial group
for neuron_idx in range(len(align_psth_Saccade)):
    angSuccess_temp = angsSuccess[np.where(distractor == 0.0)[0]]
    psth_temp = align_psth_Saccade[neuron_idx,:,:]
    # Average across trials for each angSuccess group
    group_means = []
    for group in np.unique(angSuccess_temp):
        group_indices = np.where(angSuccess_temp == group)[0]
        if len(group_indices) > 0:
            group_data = np.mean(psth_temp[group_indices], axis=0)
        else:
            group_data = np.nan
        group_means.append(group_data)
    group_means = np.array(group_means)
    if np.any(np.isnan(group_means)):
        continue  # Skip this neuron if data is incomplete
    mean_psth = group_means
    # Plotting
    plt.figure(figsize=(10, 6))
    for i, group in enumerate(np.unique(angSuccess_temp)):
        plt.plot(np.arange(-SACC_PRE, SACC_POST, S["bin_ms"]), mean_psth[i], label=f'Group {group}')
    plt.axvline(0, color='k', linestyle='--')
    plt.title(f'Neuron {good_neurons[neuron_idx]} - PSTH Aligned to Saccade')
    plt.xlabel('Time from Saccade (ms)')
    plt.ylabel('Firing Rate (Hz)')
    plt.legend()
    # Create a new directory to save the plots
    output_dir = "psth_aligned_to_saccade"
    output_path = os.path.join(out_dir, output_dir)
    os.makedirs(output_path, exist_ok=True)
    plt.savefig(os.path.join(output_path, f'neuron_{good_neurons[neuron_idx]}_psth_saccade_0.png'))
    plt.close()






Plot 45 distractor

In [ ]:

# Plot data aligned to saccade for each neuron and trial averaged across each angSuccess trial group
for neuron_idx in range(len(align_psth_Saccade45)):
    angSuccess_temp = angsSuccess[np.where(distractor == 45.0)[0]]
    psth_temp = align_psth_Saccade45[neuron_idx,:,:]
    # Average across trials for each angSuccess group
    group_means = []
    for group in np.unique(angSuccess_temp):
        group_indices = np.where(angSuccess_temp == group)[0]
        if len(group_indices) > 0:
            group_data = np.mean(psth_temp[group_indices], axis=0)
        else:
            group_data = np.nan
        group_means.append(group_data)
    group_means = np.array(group_means)
    if np.any(np.isnan(group_means)):
        continue  # Skip this neuron if data is incomplete
    mean_psth = group_means
    # Plotting
    plt.figure(figsize=(10, 6))
    for i, group in enumerate(np.unique(angSuccess_temp)):
        plt.plot(np.arange(-SACC_PRE, SACC_POST, S["bin_ms"]), mean_psth[i], label=f'Group {group}')
    plt.axvline(0, color='k', linestyle='--')
    plt.title(f'Neuron {good_neurons[neuron_idx]} - PSTH Aligned to Saccade')
    plt.xlabel('Time from Saccade (ms)')
    plt.ylabel('Firing Rate (Hz)')
    plt.legend()
    # Create a new directory to save the plots
    output_dir = "psth_aligned_to_saccade"
    output_path = os.path.join(out_dir, output_dir)
    os.makedirs(output_path, exist_ok=True)
    plt.savefig(os.path.join(output_path, f'neuron_{good_neurons[neuron_idx]}_psth_saccade_45.png'))
    plt.close()



Plot 180 degree distractor

In [ ]:

# Plot data aligned to saccade for each neuron and trial averaged across each angSuccess trial group
for neuron_idx in range(len(align_psth_Saccade180)):
    angSuccess_temp = angsSuccess[np.where(distractor == 180.0)[0]]
    psth_temp = align_psth_Saccade180[neuron_idx,:,:]
    #print(psth_temp.shape)
    # Average across trials for each angSuccess group
    group_means = []
    for group in np.unique(angSuccess_temp):
        group_indices = np.where(angSuccess_temp == group)[0]
        if len(group_indices) > 0:
            group_data = np.nanmean(psth_temp[group_indices], axis=0)
        else:
            group_data = np.nan
        group_means.append(group_data)
    group_means = np.array(group_means)
    if np.any(np.isnan(group_means)):
        continue  # Skip this neuron if data is incomplete
    mean_psth = group_means
    # Plotting
    plt.figure(figsize=(10, 6))
    for i, group in enumerate(np.unique(angSuccess_temp)):
        plt.plot(np.arange(-SACC_PRE, SACC_POST, S["bin_ms"]), mean_psth[i], label=f'Group {group}')
    plt.axvline(0, color='k', linestyle='--')
    plt.title(f'Neuron {good_neurons[neuron_idx]} - PSTH Aligned to Saccade')
    plt.xlabel('Time from Saccade (ms)')
    plt.ylabel('Firing Rate (Hz)')
    plt.legend()
    # Create a new directory to save the plots
    output_dir = "psth_aligned_to_saccade"
    output_path = os.path.join(out_dir, output_dir)
    os.makedirs(output_path, exist_ok=True)
    plt.savefig(os.path.join(output_path, f'neuron_{good_neurons[neuron_idx]}_psth_saccade_180.png'))
    plt.close()



## MI calculated for motor epoch

In [ ]:
motor_window = slice(*S["motor_bins"]) # saccade-aligned bins, set in the config

In [ ]:
# Stacking PSTH for all 0, 45, and 180
align_psth_combined = np.concatenate([align_psth_Saccade, align_psth_Saccade45, align_psth_Saccade180], axis = 1)
psth_comb_motor = np.nanmean(align_psth_combined[:,:,motor_window], axis = 2)

results_combined = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(neuron_mi_perm)(psth_comb_motor[i], angsSuccess, n_neighbors=N_NEIGHBORS, n_permutations=N_PERMUTATIONS)
    for i in range(align_psth_combined.shape[0])
)



## MI calculated for motor vs baseline epoch

In [ ]:


results_combined_comp = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(mi_diff_permutation_test)(psth_comb_baseline[i],psth_comb_motor[i], angsSuccess, n_neighbors=N_NEIGHBORS, n_permutations=N_PERMUTATIONS)
    for i in range(psth_combined.shape[0])
)


In [ ]:
# Unpack results
mi_comb_mot, p_comb_mot = zip(*results_combined)
mi_comb_mot = np.array(mi_comb_mot)
p_comb_mot = np.array(p_comb_mot)

# Unpack results
mi_comp_mot, p_comp_mot = zip(*results_combined_comp)
mi_comp_mot = np.array(mi_comp_mot)
p_comp_mot = np.array(p_comp_mot)

motor_neurons_all = np.array(np.where( (p_comb_mot < ALPHA)  | (p_comp_mot < ALPHA))[0])
print(f"Number of motor neurons: {motor_neurons_all.shape[0]}")

# Find visual neurons that have a spiking rate of less than 5 Hz in the visual window and delete them from both 180 and 45 degree visual neurons
motor_neurons_psth = psth_combined[motor_neurons_all, : ,: ]
mean_fr = np.nanmean(np.nanmean(motor_neurons_psth[:, :, motor_window], axis=2), axis = 1)

motor_neurons_final_idx = np.where((mean_fr > MIN_RATE))[0]
print(f"Number of visual neurons (> MIN_RATE): {motor_neurons_final_idx.shape[0]}")

motor_neurons_final = motor_neurons_all[motor_neurons_final_idx]

In [ ]:
motor_neurons_psth_0_new = align_psth_Saccade[motor_neurons_final,:,:]
motor_neurons_psth_45_new = align_psth_Saccade45[motor_neurons_final,:,:]
motor_neurons_psth_180_new = align_psth_Saccade180[motor_neurons_final,:,:]

## Plot all the motor neurons for each distractor type

In [ ]:
#Plot only the visual neurons psth for 45 and 180 degrees
#Plot them as 2 different subplots for each neuron. And also plot each direction in a different color
# Make sure that the firing rate is equal on both the subplots such that the y-axis has the value of the maximum firing rate between the two subplots
from pathlib import Path
bin_edges = np.arange(-SACC_PRE, SACC_POST, S["bin_ms"])
for neuron_idx in range(len(motor_neurons_final)):
    print(neuron_idx)
    neuron_id = motor_neurons_final[neuron_idx]
    print(f'Plotting PSTH for Neuron {neuron_id}...')
    psth_0_temp = motor_neurons_psth_0_new[neuron_idx,:,:]
    psth_45_temp = motor_neurons_psth_45_new[neuron_idx,:, :]
    psth_180_temp = motor_neurons_psth_180_new[neuron_idx,:, :]

    # Calculate the maximum firing rate across all groups for both distractors
    max_firing_rate_0 = max([np.max(np.mean(psth_0_temp[np.where(angsSuccess[np.where(distractor == 0.0)[0]] == group)[0], :], axis=0)) 
                              for group in np.unique(angsSuccess[np.where(distractor == 0.0)[0]])])
    max_firing_rate_45 = max([np.max(np.mean(psth_45_temp[np.where(angsSuccess[np.where(distractor == 45.0)[0]] == group)[0], :], axis=0)) 
                              for group in np.unique(angsSuccess[np.where(distractor == 45.0)[0]])])
    max_firing_rate_180 = max([np.max(np.mean(psth_180_temp[np.where(angsSuccess[np.where(distractor == 180.0)[0]] == group)[0], :], axis=0)) 
                               for group in np.unique(angsSuccess[np.where(distractor == 180.0)[0]])])
    max_firing_rate = max(max_firing_rate_0,max_firing_rate_45, max_firing_rate_180)

    plt.figure(figsize=(12, 6))
    # Plot for 0° distractor
    plt.subplot(1, 3, 1)
    for group in np.unique(angsSuccess[np.where(distractor == 0.0)[0]]):
        group_indices = np.where(angsSuccess[np.where(distractor == 0.0)[0]] == group)[0]
        if len(group_indices) > 0:
            group_mean = np.mean(psth_0_temp[group_indices, :], axis=0)
            plt.plot(bin_edges, group_mean, label=f'Group {group}')
    plt.title(f'Neuron {neuron_id} - PSTH for 0° Distractor')
    plt.xlabel('Time (ms)')
    plt.ylabel('Firing Rate (Hz)')
    plt.ylim(0, max_firing_rate + 5)  # Set y-axis limit
    plt.axvspan(0, 50, color='gray', alpha=0.3)
    plt.legend()

    # Plot for 45° distractor
    plt.subplot(1, 3, 2)
    for group in np.unique(angsSuccess[np.where(distractor == 45.0)[0]]):
        group_indices = np.where(angsSuccess[np.where(distractor == 45.0)[0]] == group)[0]
        if len(group_indices) > 0:
            group_mean = np.mean(psth_45_temp[group_indices, :], axis=0)
            plt.plot(bin_edges, group_mean, label=f'Group {group}')
    plt.title(f'Neuron {neuron_id} - PSTH for 45° Distractor')
    plt.xlabel('Time (ms)')
    plt.ylabel('Firing Rate (Hz)')
    plt.ylim(0, max_firing_rate + 5)  # Set y-axis limit
    plt.axvspan(0, 50, color='gray', alpha=0.3)
    plt.legend()

    # Plot for 180° distractor
    plt.subplot(1, 3, 3)
    for group in np.unique(angsSuccess[np.where(distractor == 180.0)[0]]):
        group_indices = np.where(angsSuccess[np.where(distractor == 180.0)[0]] == group)[0]
        if len(group_indices) > 0:
            group_mean = np.mean(psth_180_temp[group_indices, :], axis=0)
            plt.plot(bin_edges, group_mean, label=f'Group {group}')
    plt.title(f'Neuron {neuron_id} - PSTH for 180° Distractor ')
    plt.xlabel('Time (ms)')
    plt.ylabel('Firing Rate (Hz)')
    plt.ylim(0, max_firing_rate + 5)  # Set y-axis limit
    plt.axvspan(0, 50, color='gray', alpha=0.3)
    plt.legend()
    # Save the figure
    plt.tight_layout()
    new_path = Path(out_dir) / 'PSTH_Motor_neurons_all'
    new_path.mkdir(parents=True, exist_ok=True)
    plt.savefig(new_path / f'VNeuron_{neuron_id}_PSTH_all.png', dpi=300)

    plt.show()

    plt.close()


## Individual MI for motor vs baseline epoch

In [ ]:


psth_0 = np.nanmean(align_psth_Saccade[:,:, motor_window], axis = 2)
psth_45 = np.nanmean(align_psth_Saccade45[:,:, motor_window], axis=2)
psth_180 = np.nanmean(align_psth_Saccade180[:,:, motor_window], axis=2)
n_neurons, n_trials = psth_180.shape

results_0 = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(mi_diff_permutation_test)(psth_0_baseline[i], psth_0[i], angsSuccess[np.where(distractor == 0.0)[0]], n_neighbors=N_NEIGHBORS, n_permutations=N_PERMUTATIONS)
    for i in range(psth_0.shape[0])
)
# psth_180 shape: (n_neurons, n_trials)
# angs_45 shape: (n_trials,)
results_45 = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(mi_diff_permutation_test)(psth_45_baseline[i], psth_45[i], angsSuccess[np.where(distractor == 45.0)[0]], n_neighbors=N_NEIGHBORS, n_permutations=N_PERMUTATIONS)
    for i in range(psth_45.shape[0])
)

results_180 = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(mi_diff_permutation_test)(psth_180_baseline[i], psth_180[i], angsSuccess[np.where(distractor == 180.0)[0]], n_neighbors=N_NEIGHBORS, n_permutations=N_PERMUTATIONS)
    for i in range(psth_180.shape[0])
)
# Unpack results
mi_0_motcomp, p_0_motcomp = zip(*results_0)
mi_0_motcomp = np.array(mi_0_motcomp)
p_0_motcomp = np.array(p_0_motcomp)

# Unpack results
mi_45_motcomp, p_45_motcomp = zip(*results_45)
mi_45_motcomp = np.array(mi_45_motcomp)
p_45_motcomp = np.array(p_45_motcomp)

# Unpack results
mi_180_motcomp, p_180_motcomp = zip(*results_180)
mi_180_motcomp = np.array(mi_180_motcomp)
p_180_motcomp = np.array(p_180_motcomp)


## Individual MI for motor epoch

In [ ]:

results_0 = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(neuron_mi_perm)(psth_0[i], angsSuccess[np.where(distractor == 0.0)[0]], n_neighbors=N_NEIGHBORS, n_permutations=N_PERMUTATIONS)
    for i in range(psth_0.shape[0])
)
# psth_180 shape: (n_neurons, n_trials)
# angs_45 shape: (n_trials,)
results_45 = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(neuron_mi_perm)(psth_45[i], angsSuccess[np.where(distractor == 45.0)[0]], n_neighbors=N_NEIGHBORS, n_permutations=N_PERMUTATIONS)
    for i in range(psth_45.shape[0])
)

results_180 = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(neuron_mi_perm)(psth_180[i], angsSuccess[np.where(distractor == 180.0)[0]], n_neighbors=N_NEIGHBORS, n_permutations=N_PERMUTATIONS)
    for i in range(psth_45.shape[0])
)

# Unpack results
mi_0_mot, p_0_mot = zip(*results_45)
mi_0_mot = np.array(mi_0_mot)
p_0_mot = np.array(p_0_mot)

# Unpack results
mi_45_mot, p_45_mot = zip(*results_45)
mi_45_mot = np.array(mi_45_mot)
p_45_mot = np.array(p_45_mot)

# Unpack results
mi_180_mot, p_180_mot = zip(*results_180)
mi_180_mot = np.array(mi_180_mot)
p_180_mot = np.array(p_180_mot)


In [ ]:
tau_values = sio.loadmat(os.path.join(filepath, S["tau_name"]))['tau_values'][0]

In [ ]:
# Storing results to access later
import pandas as pd
results_MI_Timescale  = pd.DataFrame({
    'good_neuron_number': good_neurons,
    'psth_0': list(psth_0_mat),
    'psth_45': list(psth_45_mat),
    'psth_180': list(psth_180_mat),
    'psth_0_motor' : list(align_psth_Saccade),
    'psth_45_motor': list(align_psth_Saccade45),
    'psth_180_motor' : list(align_psth_Saccade180),
    'Timescale_ms': tau_values,
    'mi_comb': list(zip(mi_comb, p_comb)),
    'mi_comp': list(zip(mi_comp, p_comp)),
    'Visual_neurons_combined': np.isin(np.arange(n_neurons), visual_neurons_final).astype(int),
    'mi_comb_mem':list(zip(mi_comb_mem, p_comb_mem)),
    'mi_comp_mem': list(zip(mi_comp_mem, p_comp_mem)),
    'Memory_neurons_combined': np.isin(np.arange(n_neurons), memory_neurons_final).astype(int),
    'mi_comb_motor': list(zip(mi_comb_mot, p_comb_mot)),
    'mi_comp_motor':list(zip(mi_comp_mot, p_comp_mot)),
    'Motor_neurons_combined': np.isin(np.arange(n_neurons), motor_neurons_final).astype(int),
    'mi_v_0': list(zip(mi_0, p_0)),
    'mi_vcomp_0': list(zip(mi_0_vcomp, p_0_vcomp)),
    'mi_v_45': list(zip(mi_45, p_45)),
    'mi_vcomp_45': list(zip(mi_45_vcomp, p_45_vcomp)),
    'mi_v_180': list(zip(mi_180, p_180)),
    'mi_vcomp_180': list(zip(mi_180_vcomp, p_180_vcomp)),
    'mi_mem_0' : list(zip(mi_0_memcomp, p_0_memcomp)),
    'mi_mem_comp_0': list(zip(mi_0_mem, p_0_mem)),
    'mi_mem_45' : list(zip(mi_45_memcomp, p_45_memcomp)),
    'mi_mem_comp_45': list(zip(mi_45_mem, p_45_mem)),
    'mi_mem_180' : list(zip(mi_180_memcomp, p_180_memcomp)),
    'mi_mem_comp_180': list(zip(mi_180_mem, p_180_mem)),
    'mi_mt_0' : list(zip(mi_0_motcomp, p_0_motcomp)),
    'mi_mt_comp_0': list(zip(mi_0_mot, p_0_mot)),
    'mi_mt_45' : list(zip(mi_45_motcomp, p_45_motcomp)),
    'mi_mt_comp_45': list(zip(mi_45_mot, p_45_mot)),
    'mi_mt_180' : list(zip(mi_180_motcomp, p_180_motcomp)),
    'mi_mt_comp_180': list(zip(mi_180_mot, p_180_mot))

})

In [ ]:
from scipy.io import savemat

# Convert to a MATLAB-safe dictionary
matlab_dict = {}
for col in results_MI_Timescale.columns:
    try:
        matlab_dict[col] = np.array(results_MI_Timescale[col].tolist())
    except:
        # fallback to object if not convertible
        matlab_dict[col] = results_MI_Timescale[col].values

savemat(os.path.join(out_dir, 'results_MI_Timescale' + S["output_suffix"] + '.mat'), matlab_dict)


In [ ]:
# Save as pickle (best for preserving structure)
results_MI_Timescale.to_pickle(os.path.join(out_dir, 'results_MI_Timescale' + S["output_suffix"] + '.pkl'))


# Plotting the timescale for different neuron groups

visual_ts = np.array(results_MI_Timescale['Timescale_ms'][results_MI_Timescale['Visual_neurons_combined'] == 1])
memory_ts = np.array(results_MI_Timescale['Timescale_ms'][results_MI_Timescale['Memory_neurons_combined'] == 1])
motor_ts = np.array(results_MI_Timescale['Timescale_ms'][results_MI_Timescale['Motor_neurons_combined'] == 1])

print(f'Mean Visual timescale {np.mean(visual_ts)} ms')
print(f'Mean Memory timescale {np.mean(memory_ts)} ms')
print(f'Mean Motor timescale {np.mean(motor_ts)} ms')


import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Combine all data into one long array
all_ts = np.concatenate([visual_ts, memory_ts, motor_ts])

area = 'SEF (4,4)'
monkey = 'Vennie'

# Create a corresponding label array
labels = (
    [f'Visual, n = {len(visual_ts)}'] * len(visual_ts) +
    [f'Memory, n = {len(memory_ts)}'] * len(memory_ts) +
    [f'Motor, n = {len(motor_ts)}']  * len(motor_ts)
)

# Create a DataFrame for seaborn
df = pd.DataFrame({
    'Timescale_ms': all_ts,
    'NeuronType': labels
})

# Plot violin plot
plt.figure(figsize=(8, 6))
sns.violinplot(x='NeuronType', y='Timescale_ms', data=df, inner='box', palette='muted')

plt.title(f'Timescale Distributions by Neuron Type - {area} - {monkey}')
plt.ylabel('Timescale (ms)')
plt.xlabel('Neuron Type')
plt.tight_layout()
plt.show()


import numpy as np
import matplotlib.pyplot as plt

# Sort the values
visual_sorted = np.sort(visual_ts)
memory_sorted = np.sort(memory_ts)
motor_sorted  = np.sort(motor_ts)

# Compute cumulative probabilities
visual_cdf = np.linspace(0, 1, len(visual_sorted))
memory_cdf = np.linspace(0, 1, len(memory_sorted))
motor_cdf  = np.linspace(0, 1, len(motor_sorted))

# Plot CDFs
plt.figure(figsize=(8, 6))

plt.plot(visual_sorted, visual_cdf, label=f'Visual, n = {len(visual_ts)}', lw=2)
plt.plot(memory_sorted, memory_cdf, label=f'Memory, n = {len(memory_ts)}', lw=2)
plt.plot(motor_sorted, motor_cdf, label=f'Motor, n = {len(motor_ts)}', lw=2)

plt.xlabel('Timescale (ms)')
plt.ylabel('Cumulative Probability')
plt.title(f'Timescales by Neuron Type - {area} - {monkey}')
plt.legend()
plt.tight_layout()
plt.show()


import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import os
from scipy.stats import vonmises
from scipy.optimize import minimize

def compute_hwhm(kappa):
    """Compute Half-Width at Half-Maximum (HWHM) from kappa safely."""
    if kappa <= 0:
        return np.nan  # No meaningful tuning or undefined
    val = (np.log(0.5) / kappa) + 1
    val = np.clip(val, -1.0, 1.0)  # Ensure it's within domain of arccos
    hwhm_rad = np.arccos(val)
    return np.degrees(hwhm_rad)

# Define von Mises function
def von_mises(x, A, kappa, mu, B):
    # x is the angle in radians 
    norm = 2 * np.pi * vonmises(kappa).pdf(0)
    return A * np.exp(kappa * np.cos(x - mu))/norm + B

def loss_fn(params, x, y, lam=0.1):
    A, kappa, mu, B = params
    y_pred = von_mises(x, A, kappa, mu, B)
    residuals = y - y_pred
    ssr = np.sum(residuals**2)
    
    # Example regularization: penalize large kappa
    reg = lam * kappa**2
    
    return ssr + reg

# Define angles and sort order
angle = [45, 0, 315, 270, 225, 180, 135, 90]  # Target angles
# Ensure sorted_groups indices are valid for the size of the angle array
sorted_groups = np.array([i for i in [1, 0, 7, 6, 5, 4, 3, 2] if i < len(angle)])
sorted_angles = [angle[i] for i in sorted_groups]
canonical_angles = [0, 45, 90, 135, 180, 225, 270, 315]


"""
# Only for FEF recording
angle = [45, 0, 315,  225, 180, 135]  # Target angles
# Ensure sorted_groups indices are valid for the size of the angle array
sorted_groups = np.array([i for i in [1, 0, 5, 4, 3, 2] if i < len(angle)])
sorted_angles = [angle[i] for i in sorted_groups]
canonical_angles = [0, 45, 135, 180, 225, 315]
"""



psth_path = r'D:\Neuropixel_data\061125_Vennie\merged_all\supercat_MGS2_g0\psth_new'
sorted_groups = np.array([i for i in [1, 0, 7, 6, 5, 4, 3, 2] if i < len(angle)])
sorted_angles = [angle[i] for i in sorted_groups]
canonical_angles = [0, 45, 90, 135, 180, 225, 270, 315]


"""
# Only for FEF recording
angle = [45, 0, 315,  225, 180, 135]  # Target angles
# Ensure sorted_groups indices are valid for the size of the angle array
sorted_groups = np.array([i for i in [1, 0, 5, 4, 3, 2] if i < len(angle)])
sorted_angles = [angle[i] for i in sorted_groups]
canonical_angles = [0, 45, 135, 180, 225, 315]
"""

canonical_rad = np.deg2rad(canonical_angles)
angles_rad = np.deg2rad(sorted_angles)

# Output directory for plots
output_dir = "tuning_curves_new"
output_path = os.path.join(psth_path, output_dir)
os.makedirs(output_path, exist_ok=True)

# Initialize dictionary to store neuron data
neuron_data_45 = {}

# Loop through neurons
for neuron_idx in range(min(len(visual_neurons_psth_45_new), len(visual_neurons_final))):
    # Extract visual epoch: (trials, timepoints)
    visual_epoch_0 = visual_neurons_psth_45_new[neuron_idx, :, visual_window]  # (neurons, trials, time)
    baseline_epoch_0 = visual_neurons_psth_45_new[neuron_idx, :, :50]  # (neurons, trials, time)
    angs_0 = angsSuccess[np.where(distractor == 45.0)[0]]

    group_means = []
    group_means_baseline = []
    for group in np.unique(angs_0):
        group_indices = np.where(angs_0 == group)[0]
        #group_indices = np.where((distractor == 45.0) & (angsSuccess == group))[0]
        
        if len(group_indices) > 0:
            # Mean spike rate across trials and timepoints
            group_data = np.mean(visual_epoch_0[group_indices], axis=(0, 1))
            group_data_baseline = np.mean(baseline_epoch_0[group_indices], axis=(0, 1))
            # Max firing rate
        else:
            group_data = np.nan
        group_means.append(group_data)
        group_means_baseline.append(group_data_baseline)
        
    group_means = np.array(group_means)
    group_means_baseline = np.array(group_means_baseline)
    if np.any(np.isnan(group_means)):
        continue  # Skip this neuron if data is incomplete

    mean_visual_0 = group_means
    sorted_data_0 = mean_visual_0[sorted_groups]
    sorted_data_baseline_0 = group_means_baseline[sorted_groups]


    # Fit von Mises model
    try:
        pref_dir_guess = np.deg2rad(sorted_angles[np.argmax(sorted_data_0)])
        p0 = [np.max(sorted_data_0) - np.min(sorted_data_0), 1, pref_dir_guess, np.min(sorted_data_baseline_0)]

        # Bounds: (A, kappa, mu, B)
        bounds = [(0, None), (0, 10), (0, 2*np.pi), (0, None)]

        # Fit with minimize
        res = minimize(loss_fn, p0, args=(angles_rad, sorted_data_0, 1), bounds=bounds, method='L-BFGS-B')

        # Extract fit params
        A_fit, kappa_fit, mu_fit, B_fit = res.x
        hwhm = compute_hwhm(kappa_fit)
        fwhm = hwhm * 2

        # Generate fit
        theta_fit = np.linspace(0, 2 * np.pi, 500)
        fit_vals = von_mises(theta_fit, A_fit, kappa_fit, mu_fit, B_fit)

        # Interpolate fit values at canonical angles
        canonical_data = von_mises(canonical_rad, A_fit, kappa_fit, mu_fit, B_fit)



        # Compute R² for minimize fit
        y_actual = sorted_data_0
        y_pred = von_mises(angles_rad, A_fit, kappa_fit, mu_fit, B_fit)
        ss_tot = np.sum((y_actual - np.mean(y_actual)) ** 2)
        ss_res = np.sum((y_actual - y_pred) ** 2)
        r2 = 1 - (ss_res / ss_tot)


        fig = plt.figure(figsize=(12, 5))
        ax_polar = fig.add_subplot(1, 2, 1, polar=True)
        ax_cart = fig.add_subplot(1, 2, 2)


        # Polar plot
       
        ax_polar.plot(angles_rad, sorted_data_0, 'o', label='Data')
        ax_polar.plot(theta_fit, fit_vals, '-', color='red', label='Von Mises Fit')
        ax_polar.plot([mu_fit, mu_fit], [0, max(fit_vals)], 'k--', label='Preferred Angle')
        ax_polar.set_title(f'Neuron {visual_neurons_final[neuron_idx]}\nTuning Curve (Polar), R2 = {r2:.3f}')
        ax_polar.legend(loc='lower right')

        # Cartesian plot
        
        # Cartesian plot
        
        ax_cart.plot(np.rad2deg(angles_rad), sorted_data_0, 'o', label='Data')
        ax_cart.plot(np.rad2deg(theta_fit), fit_vals, '-', color='red', label='Von Mises Fit')
        ax_cart.set_xticks([0, 45, 90, 135, 180, 225, 270, 315])
        ax_cart.set_xlim(0, 360)
        ax_cart.set_xlabel('Angle (degrees)')
        ax_cart.set_ylabel('Firing Rate (Hz)')
        ax_cart.set_title(f'Tuning Curve (Cartesian)\nFWHM: {fwhm:.2f}°, mi45: {visual_neurons_mi45[neuron_idx]:.2f}')
        ax_cart.legend()
        ax_cart.grid(True)

        plt.tight_layout()
        fname = os.path.join(output_path, f"neuron_{visual_neurons_final[neuron_idx]}_tuning_45.png")
        plt.savefig(fname)
        plt.close()

    except RuntimeError as e:
        print(f"Fit failed for neuron {visual_neurons_final[neuron_idx]}: {e}")
        continue


#psth_path = r'D:\Neuropixel_data\061025_Vennie\merged_all\supercat_MGS1_g0\psth_new'
sorted_groups = np.array([i for i in [1, 0, 7, 6, 5, 4, 3, 2] if i < len(angle)])
sorted_angles = [angle[i] for i in sorted_groups]
canonical_angles = [0, 45, 90, 135, 180, 225, 270, 315]


"""
# Only for FEF recording
angle = [45, 0, 315,  225, 180, 135]  # Target angles
# Ensure sorted_groups indices are valid for the size of the angle array
sorted_groups = np.array([i for i in [1, 0, 5, 4, 3, 2] if i < len(angle)])
sorted_angles = [angle[i] for i in sorted_groups]
canonical_angles = [0, 45, 135, 180, 225, 315]
"""

canonical_rad = np.deg2rad(canonical_angles)
angles_rad = np.deg2rad(sorted_angles)

# Output directory for plots
output_dir = "tuning_curves_new"
output_path = os.path.join(psth_path, output_dir)
os.makedirs(output_path, exist_ok=True)

# Initialize dictionary to store neuron data
neuron_data_180 = {}

# Loop through neurons
for neuron_idx in range(min(len(visual_neurons_psth_180_new), len(visual_neurons_final))):
    # Extract visual epoch: (trials, timepoints)
    visual_epoch_0 = visual_neurons_psth_180_new[neuron_idx, :, visual_window]  # (neurons, trials, time)
    baseline_epoch_0 = visual_neurons_psth_180_new[neuron_idx, :, :50]  # (neurons, trials, time)
    angs_0 = angsSuccess[np.where(distractor == 180.0)[0]]

    group_means = []
    group_means_baseline = []
    for group in np.unique(angs_0):
        group_indices = np.where(angs_0 == group)[0]
        #group_indices = np.where((distractor == 45.0) & (angsSuccess == group))[0]
        
        if len(group_indices) > 0:
            # Mean spike rate across trials and timepoints
            group_data = np.mean(visual_epoch_0[group_indices], axis=(0, 1))
            group_data_baseline = np.mean(baseline_epoch_0[group_indices], axis=(0, 1))
            # Max firing rate
        else:
            group_data = np.nan
        group_means.append(group_data)
        group_means_baseline.append(group_data_baseline)
        
    group_means = np.array(group_means)
    group_means_baseline = np.array(group_means_baseline)
    if np.any(np.isnan(group_means)):
        continue  # Skip this neuron if data is incomplete

    mean_visual_0 = group_means
    sorted_data_0 = mean_visual_0[sorted_groups]
    sorted_data_baseline_0 = group_means_baseline[sorted_groups]


    # Fit von Mises model
    try:
        pref_dir_guess = np.deg2rad(sorted_angles[np.argmax(sorted_data_0)])
        p0 = [np.max(sorted_data_0) - np.min(sorted_data_0), 1, pref_dir_guess, np.min(sorted_data_baseline_0)]

        # Bounds: (A, kappa, mu, B)
        bounds = [(0, None), (0, 10), (0, 2*np.pi), (0, None)]

        # Fit with minimize
        res = minimize(loss_fn, p0, args=(angles_rad, sorted_data_0, 1), bounds=bounds, method='L-BFGS-B')

        # Extract fit params
        A_fit, kappa_fit, mu_fit, B_fit = res.x
        hwhm = compute_hwhm(kappa_fit)
        fwhm = hwhm * 2

        # Generate fit
        theta_fit = np.linspace(0, 2 * np.pi, 500)
        fit_vals = von_mises(theta_fit, A_fit, kappa_fit, mu_fit, B_fit)

        # Interpolate fit values at canonical angles
        canonical_data = von_mises(canonical_rad, A_fit, kappa_fit, mu_fit, B_fit)



      

        y_actual = sorted_data_0
        y_pred = von_mises(angles_rad, A_fit, kappa_fit, mu_fit, B_fit)
        ss_tot = np.sum((y_actual - np.mean(y_actual)) ** 2)
        ss_res = np.sum((y_actual - y_pred) ** 2)
        r2 = 1 - (ss_res / ss_tot)

        fig = plt.figure(figsize=(12, 5))
        ax_polar = fig.add_subplot(1, 2, 1, polar=True)
        ax_cart = fig.add_subplot(1, 2, 2)


        # Polar plot
       
        ax_polar.plot(angles_rad, sorted_data_0, 'o', label='Data')
        ax_polar.plot(theta_fit, fit_vals, '-', color='red', label='Von Mises Fit')
        ax_polar.plot([mu_fit, mu_fit], [0, max(fit_vals)], 'k--', label='Preferred Angle')
        ax_polar.set_title(f'Neuron {visual_neurons_final[neuron_idx]}\nTuning Curve (Polar), R2 = {r2:.3f}')
        ax_polar.legend(loc='lower right')

        # Cartesian plot
        
        # Cartesian plot
        
        ax_cart.plot(np.rad2deg(angles_rad), sorted_data_0, 'o', label='Data')
        ax_cart.plot(np.rad2deg(theta_fit), fit_vals, '-', color='red', label='Von Mises Fit')
        ax_cart.set_xticks([0, 45, 90, 135, 180, 225, 270, 315])
        ax_cart.set_xlim(0, 360)
        ax_cart.set_xlabel('Angle (degrees)')
        ax_cart.set_ylabel('Firing Rate (Hz)')
        ax_cart.set_title(f'Tuning Curve (Cartesian)\nFWHM: {fwhm:.2f}°, mi180: {visual_neurons_mi180[neuron_idx]:.2f}')
        ax_cart.legend()
        ax_cart.grid(True)

        plt.tight_layout()
        fname = os.path.join(output_path, f"neuron_{visual_neurons_final[neuron_idx]}_tuning_180.png")
        plt.savefig(fname)
        plt.close()

    except RuntimeError as e:
        print(f"Fit failed for neuron {visual_neurons_final[neuron_idx]}: {e}")
        continue


#Plot only the visual neurons psth for 45 and 180 degrees
#Plot them as 2 different subplots for each neuron. And also plot each direction in a different color
# Make sure that the firing rate is equal on both the subplots such that the y-axis has the value of the maximum firing rate between the two subplots
from pathlib import Path
window = [-0.5, 1.8]
bin_width = 0.01  # 10 ms bin width
bin_edges = np.arange(window[0], window[1] + bin_width, bin_width)
bin_edges = bin_edges[:-1]  # Center the bins
for neuron_idx in range(len(visual_neurons_final)):
    print(neuron_idx)
    neuron_id = visual_neurons_final[neuron_idx]
    print(f'Plotting PSTH for Neuron {neuron_id}...')
    psth_45_temp = visual_neurons_psth_45_new[neuron_idx,:, :]
    psth_180_temp = visual_neurons_psth_180_new[neuron_idx,:, :]

    # Calculate the maximum firing rate across all groups for both distractors
    max_firing_rate_45 = max([np.max(np.mean(psth_45_temp[np.where(angsSuccess[np.where(distractor == 45.0)[0]] == group)[0], :], axis=0)) 
                              for group in np.unique(angsSuccess[np.where(distractor == 45.0)[0]])])
    max_firing_rate_180 = max([np.max(np.mean(psth_180_temp[np.where(angsSuccess[np.where(distractor == 180.0)[0]] == group)[0], :], axis=0)) 
                               for group in np.unique(angsSuccess[np.where(distractor == 180.0)[0]])])
    max_firing_rate = max(max_firing_rate_45, max_firing_rate_180)

    plt.figure(figsize=(12, 6))

    # Plot for 45° distractor
    plt.subplot(1, 2, 1)
    for group in np.unique(angsSuccess[np.where(distractor == 45.0)[0]]):
        group_indices = np.where(angsSuccess[np.where(distractor == 45.0)[0]] == group)[0]
        if len(group_indices) > 0:
            group_mean = np.mean(psth_45_temp[group_indices, :], axis=0)
            plt.plot(bin_edges * 1000, group_mean, label=f'Group {group}')
    plt.title(f'Neuron {neuron_id} - PSTH for 45° Distractor, MI: {visual_neurons_mi45[neuron_idx]:.2f}')
    plt.xlabel('Time (ms)')
    plt.ylabel('Firing Rate (Hz)')
    plt.ylim(0, max_firing_rate + 5)  # Set y-axis limit
    plt.axvspan(0, 50, color='gray', alpha=0.3)
    plt.legend()

    # Plot for 180° distractor
    plt.subplot(1, 2, 2)
    for group in np.unique(angsSuccess[np.where(distractor == 180.0)[0]]):
        group_indices = np.where(angsSuccess[np.where(distractor == 180.0)[0]] == group)[0]
        if len(group_indices) > 0:
            group_mean = np.mean(psth_180_temp[group_indices, :], axis=0)
            plt.plot(bin_edges * 1000, group_mean, label=f'Group {group}')
    plt.title(f'Neuron {neuron_id} - PSTH for 180° Distractor , MI: {visual_neurons_mi180[neuron_idx]:.2f}')
    plt.xlabel('Time (ms)')
    plt.ylabel('Firing Rate (Hz)')
    plt.ylim(0, max_firing_rate + 5)  # Set y-axis limit
    plt.axvspan(0, 50, color='gray', alpha=0.3)
    plt.legend()
    # Save the figure
    plt.tight_layout()
    new_path = Path(psth_path) / 'PSTH_Visual_neurons'
    new_path.mkdir(parents=True, exist_ok=True)
    plt.savefig(new_path / f'VNeuron_{neuron_id}_PSTH_45_vs_180.png', dpi=300)

    plt.show()

    plt.close()


#  Kolmogorov-Smirnov (K-S) test on the mutual information values
from scipy.stats import ks_2samp

# Find the indices of visual_neurons_final in visual_neurons
indices_in_visual_neurons = np.where(np.isin(visual_neurons, visual_neurons_final))[0]

# Use these indices to extract the corresponding mutual information values
visual_neurons_mi180 = mi_180[visual_neurons_idx[indices_in_visual_neurons]]
visual_neurons_mi45 = mi_45[visual_neurons_idx[indices_in_visual_neurons]]

stat, p_value_v = ks_2samp(visual_neurons_mi180,visual_neurons_mi45)
print(f'K-S test statistic visual: {stat}, p-value: {p_value_v}')


psth_path = Path(r'D:\Neuropixel_data\061125_Vennie\merged_all\supercat_MGS2_g0\psth_new')
# Save the mutual information values and p-values for visual neurons
# Visualize the distribution of mi_45 and mi_180 for all the visual neurons as a cdf
plt.figure(figsize=(10, 6))
plt.hist(visual_neurons_mi180, bins=30, alpha=0.5, label='mi_180', color='blue', density=True, cumulative=True)
plt.hist(visual_neurons_mi45, bins=30, alpha=0.5, label='mi_45', color='orange', density=True, cumulative=True)
plt.legend()
plt.xlabel('Mutual Information')
plt.ylabel('Cumulative Density')
plt.title(f'Cumulative Distribution of Mutual Information for Visual Neurons p-value: {p_value_v}')

plt.savefig(psth_path / 'mi_visual_neurons_cdf.png', dpi=300)
plt.show()